In [1]:
from tscglue.utils import load_s3_parquet_cached
import polars as pl

res = load_s3_parquet_cached(skip_empty=True)
print(f"Loaded {len(res)} rows across {res['dataset'].n_unique()} datasets and {res['model'].n_unique()} models")

Skipped 31 empty parquet file(s) on S3
Loaded 198446 rows across 112 datasets and 76 models


In [2]:
# Runs per model
res.group_by(["dataset", "model"]).agg(pl.len().alias("n_folds")).group_by("model").agg(
    pl.col("n_folds").sum().alias("n_rows"),
    pl.col("dataset").n_unique().alias("n_datasets"),
    pl.col("n_folds").mean().alias("avg_folds_per_dataset"),
).sort("model")

model,n_rows,n_datasets,avg_folds_per_dataset
str,u32,u32,f64
"""TSCGlue-17-4-26""",3360,112,30.0
"""TSCGlue-17-4-26-c2""",1064,106,10.037736
"""TSCGlue-17-4-26-c3""",1003,107,9.373832
"""TSCGlue-17-4-26-c5""",482,105,4.590476
"""TSCGlue-17-4-26-r2""",918,106,8.660377
…,…,…,…
"""rstsf-random-ridge""",3360,112,30.0
"""rstsf-unsupervised""",2044,112,18.25
"""rstsf-unsupervised-raw""",1215,112,10.848214


In [3]:
# Export to parquet
out_path = "zenodo_results.parquet"
res.write_parquet(out_path)
print(f"Exported to {out_path}")

Exported to zenodo_results.parquet
